# Coupling RiverBedDynamics and OverlandFlow — New Features Tutorial

This tutorial extends the introductory RiverBedDynamics tutorial by demonstrating the new capabilities added to the component. The physical scenario is identical: a 1500 m gravel-bed channel receiving an excess sediment supply, run for one day. This provides a direct baseline for comparing how each new feature changes the result.

**What is new and demonstrated here:**

| Feature | Parameter | Section |
|---|---|---|
| Link-depth synchronisation fix | `map_mean_of_link_nodes_to_link` in loop | 4 |
| CFL stability analysis | `calc_max_stable_dt()` | 5 |
| Gravitational bed diffusion | `use_bed_diffusion`, `bed_diffusion_mu` | 6 |
| RK2 higher-accuracy integrator | `time_stepping="rk2"` | 7 |
| Post-run numerical health check | `_bed_surf__gsd_residual_max` | 8 |

Sections 1–4 follow the same structure as the introductory tutorial. Sections 5–8 are new.

---
## 1. Imports

In [ ]:
import copy

import matplotlib.cm as cm
import numpy as np
from IPython.display import clear_output, display
from matplotlib import pyplot as plt

from landlab import imshow_grid
from landlab.components import OverlandFlow, RiverBedDynamics
from landlab.grid.mappers import map_mean_of_link_nodes_to_link
from landlab.io.esri_ascii import load as load_esri_ascii

---
## 2. Parameters

All parameters are identical to the introductory tutorial. `max_dt = 1 s` is used throughout. Section 5 explains how to determine this value analytically using the new CFL utilities.

In [ ]:
# ASCII raster DEM containing the bed surface elevation
zDEM = "bedElevationDEM.asc"

# ASCII file containing grain size distribution
gsd = np.loadtxt("bed_gsd.txt")

# Elapsed time (sec)
t = 0

# Elapsed time when getting initial conditions (sec)
t0 = 0

# Maximum time step in sec
# See Section 5 for how this value is determined from the CFL condition
max_dt = 1

# Maximum simulation time (sec)
sim_max_t = 86400 + max_dt  # To reach equilibrium use 60 * 86400 + max_dt

# Maximum simulation time when getting initial conditions (sec)
sim_max_t_0 = 86400 + max_dt

# Manning's n
n = 0.03874

# Bedload rate at inlet [m²/s]
in_qb = -0.0087

# Link IDs at which sediment supply enters
in_l = np.array((221, 222))

# Node IDs at which water depth is specified
in_n = np.array((129, 130))

# Node IDs for fixed-elevation outlet nodes
fixed_nodes_id = np.array((1, 2, 5, 6))

# Node IDs for zero-gradient upstream ghost cells
calc_node_id = np.arange(128, 136)

# Node IDs for longitudinal profile extraction
sample_node_id = np.arange(5, 126, 4)

# Interval at which bed profile is stored [sec]
save_data_time_interval = 86400

Analytical solution parameters (MPM equation):

In [ ]:
# Total discharge [m³/s] — negative means leftward flow
Q = -50

# Critical Shields stress
tauCrStar = 0.047

# MPM exponent
qbStar_exp = 3 / 2

# MPM coefficient
qbStar_coeff = 8

# Study domain extent [m]
x0 = 0
xf = 1500

---
## 3. Grid, OverlandFlow, and RiverBedDynamics Setup

This section is identical to the introductory tutorial.

In [ ]:
OverlandFlow.input_var_names

In [ ]:
# Load DEM — new API: load_esri_ascii returns the grid with the field already attached
with open(zDEM) as f:
    rmg = load_esri_ascii(f, name="topographic__elevation")
z = rmg.at_node["topographic__elevation"]

rmg.add_zeros("surface_water__depth", at="node")
rmg["node"]["topographic__elevation_original"] = copy.deepcopy(
    rmg["node"]["topographic__elevation"]
)

In [ ]:
imshow_grid(rmg, "topographic__elevation", vmin=0, vmax=45)

In [ ]:
of = OverlandFlow(
    rmg,
    h_init=0.001,
    mannings_n=n,
    rainfall_intensity=0.0,
)
of._rainfall_intensity = np.zeros_like(z, dtype=float)
of._rainfall_intensity[in_n] = 0.02

In [ ]:
RiverBedDynamics.input_var_names

In [ ]:
RiverBedDynamics.var_mapping

In [ ]:
rmg.add_zeros("surface_water__velocity", at="node")
rmg.add_zeros("surface_water__velocity", at="link")
rmg["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
    rmg, "surface_water__depth"
)

In [ ]:
rmg.set_watershed_boundary_condition_outlet_id([1, 2], z, 45.0)

In [ ]:
# Fixed-elevation outlet nodes
fixed_nodes = np.zeros_like(z)
fixed_nodes[fixed_nodes_id] = 1

# Prescribed sediment supply at inlet links
qb = np.full(rmg.number_of_links, 0.0)
qb[in_l] = in_qb

We instantiate `RiverBedDynamics` with `check_advective_cfl=False` because in a coupled run the timestep is controlled by `OverlandFlow`, which manages its own flow-stability CFL. The morphodynamic CFL warning is suppressed to avoid noise. When running `RiverBedDynamics` standalone, remove this argument (it defaults to `True`).

Additional new parameters demonstrated in later sections:
- `use_bed_diffusion` / `bed_diffusion_mu` — Section 6
- `time_stepping` — Section 7

In [ ]:
rbd = RiverBedDynamics(
    rmg,
    gsd=gsd,
    outlet_boundary_condition="fixedValue",
    bed_surf__elev_fix_node=fixed_nodes,
    sed_transp__bedload_rate_fix_link=qb,
    check_advective_cfl=False,
)

In [ ]:
# Ghost cells: zero-gradient BC for the upstream boundary
n_col = rmg.number_of_node_columns
n_row = rmg.number_of_node_rows

n_row_calc_n = int(calc_node_id.shape[0] / n_col)
calc_node_id = np.reshape(calc_node_id, (n_row_calc_n, n_col))

In [ ]:
# Analytical equilibrium profile (MPM)
z0 = np.reshape(z, (n_row, n_col))[1:-2, 1]
z0 = copy.deepcopy(z0)

D50    = rbd._bed_surf__median_size_node[0] / 1000   # mm → m
qbStar = np.abs(in_qb) / (np.sqrt(rbd._R * rbd._g * D50) * D50)
k1     = (np.abs(Q) * n / rmg.dx) ** (3 / 5)
k2     = k1 / (rbd._R * D50)
S      = ((1 / k2) * ((qbStar / qbStar_coeff) ** (1 / qbStar_exp) + tauCrStar)) ** (10 / 7)
x      = np.linspace(x0, xf, z0.shape[0])
zAnalytic = S * x

print(f"D50 = {D50*1000:.1f} mm")
print(f"Equilibrium slope S = {S:.6f} m/m")

---
## 4. Hydraulic Spin-up and Coupled Simulation

**Key fix vs. the original tutorial:** After every `of.overland_flow()` call, `OverlandFlow` updates `surface_water__depth` at *nodes* only. The link field must be explicitly remapped before computing velocity, otherwise links stay at `h_init = 0.001 m` and `discharge / depth` produces velocities of ~1000 m/s, which blows up the shear stress and bed elevation.

This one-line fix is required in both the spin-up loop and the main loop:
```python
rmg["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
    rmg, "surface_water__depth"
)
```

In [ ]:
progress0 = 0
while t0 < sim_max_t_0:
    of.overland_flow(dt=max_dt)

    # Remap node depths → links (required — OverlandFlow only updates nodes)
    rmg["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg, "surface_water__depth"
    )

    t0 += max_dt
    progress = int((t0 / sim_max_t_0) * 100)
    if progress > progress0 + 1:
        print(f"\rHydraulic spin-up: [{progress}%]", end="")
        progress0 = progress

rmg["link"]["surface_water__velocity"] = (
    rmg["link"]["surface_water__discharge"] / rmg["link"]["surface_water__depth"]
)
print("\nSpin-up complete.")

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))

plt.sca(axs[0])
imshow_grid(rmg, "topographic__elevation", vmin=0, vmax=45)
plt.title("Topographic elevation [m]")

plt.sca(axs[1])
imshow_grid(rmg, "surface_water__depth", cmap="Blues")
plt.title("Surface water depth [m]")

plt.tight_layout()
plt.show()

In [ ]:
def plotCurrentBedState(z_evolution, t):
    zPrevious = z_evolution[1:-1, :]
    zCurrent  = z_evolution[-1, :]
    clear_output(wait=True)
    if zPrevious.shape[0] > 0:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(x, z0, color="black", label="Initial condition")
        num_rows = zPrevious.shape[0]
        colors = cm.Greys(np.linspace(0.3, 0.7, num_rows))
        for row in zPrevious:
            ax.plot(x, row, color=colors[num_rows - 1])
            num_rows -= 1
        ax.plot(x, zCurrent, color="tab:orange",
                label=f"t = {t:.0f} s", linestyle="--")
        ax.plot(x, zAnalytic, color="blue", label="Analytical solution")
        ax.legend()
        ax.set_title("Bed elevation evolution")
        ax.set_xlabel("Streamwise distance [m]")
        ax.set_ylabel("Elevation [m]")
        ax.set_xlim(0, 1500)
        ax.set_ylim(0, 40)
        plt.tight_layout()
        display(fig)
        plt.close(fig)

Main coupled simulation loop:

In [ ]:
save_data_time_interval_original = copy.deepcopy(save_data_time_interval)
z_evolution = rmg["node"]["topographic__elevation"][sample_node_id]
progress0 = 0

while t < sim_max_t:

    # Velocity at previous time step (uses already-remapped link depths)
    rbd._surface_water__velocity_prev_time_link = (
        rmg["link"]["surface_water__discharge"] / rmg["link"]["surface_water__depth"]
    )

    of.overland_flow(dt=max_dt)

    # Remap node depths → links before computing velocity
    rmg["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg, "surface_water__depth"
    )

    rmg["link"]["surface_water__velocity"] = (
        rmg["link"]["surface_water__discharge"] / rmg["link"]["surface_water__depth"]
    )

    rbd._grid._dt = max_dt
    rbd.run_one_step()

    # Zero-gradient upstream BC
    for i in calc_node_id:
        rmg["node"]["topographic__elevation"][i] = (
            rmg["node"]["topographic__elevation"][i - n_col]
        )

    save_data_time_interval -= max_dt
    if save_data_time_interval <= 0:
        z_evolution = np.vstack(
            [z_evolution, rmg["node"]["topographic__elevation"][sample_node_id]]
        )
        plotCurrentBedState(z_evolution, t)
        save_data_time_interval = save_data_time_interval_original

    t += max_dt
    progress = int((t / sim_max_t) * 100)
    if progress > progress0 + 1:
        print(f"\rSimulation: [{progress}%]", end="")
        progress0 = progress

In [ ]:
z = rmg["node"]["topographic__elevation"]
z0_plot = rmg["node"]["topographic__elevation_original"]
fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(12, 4))

plt.sca(axs[0])
imshow_grid(rmg, z0_plot, vmin=0, vmax=45, color_for_closed="None")
plt.title("Elev. [m] at t = 0")

plt.sca(axs[1])
imshow_grid(rmg, z, vmin=0, vmax=45, color_for_closed="None")
plt.title("Elev. [m] at t = 1 day")

plt.sca(axs[2])
imshow_grid(rmg, z - z0_plot, vmin=0, vmax=20, color_for_closed="None")
plt.title("Elevation change [m]")

plt.sca(axs[3])
imshow_grid(rmg, "surface_water__depth", cmap="Blues", color_for_closed="None")
plt.title("Water depth [m]")

plt.tight_layout()
plt.show()

---
## 5. CFL Stability Analysis — How to Choose a Safe Timestep

The new component provides three methods to compute the maximum morphodynamically stable timestep from the current hydraulic and transport state:

```python
rbd.calc_max_stable_dt_advective(safety=0.5)  # advective Exner CFL
rbd.calc_max_stable_dt_diffusive(safety=0.5)  # diffusive CFL (when use_bed_diffusion=True)
rbd.calc_max_stable_dt(safety=0.5)            # min of both — use this in practice
```

The `safety` parameter accounts for non-linearity and 2-D effects. `safety=0.5` is conservative; `safety=1.0` is the theoretical limit.

**Recommended workflow for a new simulation:**
1. Run one simulation step with any small dt.
2. Call `calc_max_stable_dt()` to read the CFL limit.
3. Use that value (or a fraction of it) for the full run.

In [ ]:
# Read CFL limits at the end of the simulation
dt_adv_50  = rbd.calc_max_stable_dt_advective(safety=0.5)
dt_adv_100 = rbd.calc_max_stable_dt_advective(safety=1.0)
dt_diff    = rbd.calc_max_stable_dt_diffusive(safety=0.5)  # inf when diffusion off
dt_safe    = rbd.calc_max_stable_dt(safety=0.5)

print("=== CFL stability analysis ===")
print(f"  Advective CFL limit (safety=0.5): {dt_adv_50:.2f} s")
print(f"  Advective CFL limit (safety=1.0): {dt_adv_100:.2f} s")
print(f"  Diffusive CFL limit (safety=0.5): {dt_diff:.2f} s  "
      f"(inf = diffusion disabled)")
print()
print(f"  Recommended dt (safety=0.5): {dt_safe:.2f} s")
print(f"  Timestep used in this run:   {max_dt:.2f} s")
print()
if max_dt <= dt_safe:
    print(f"  ✅  dt={max_dt} s is within the CFL-safe limit.")
else:
    print(f"  ⚠️   dt={max_dt} s exceeds the recommended limit of {dt_safe:.2f} s.")

---
## 6. Gravitational Bed Diffusion

On a square-cell grid, the finite-volume Exner equation can produce a staircase pattern in the bed surface — small steps aligned with grid diagonals — because transport only acts along link directions. The new component adds an optional gravitational diffusion correction term (Engelund 1974; Talmon et al. 1995):

$$
(1 - \lambda_p) \frac{\partial z}{\partial t} = -\nabla \cdot q_b + \nabla \cdot (D \nabla z)
$$

The diffusion coefficient $D$ is computed as $D = |q_b| / \mu$ (nonlinear mode, default), where $\mu$ is the Talmon friction parameter `bed_diffusion_mu`. $D$ is zero wherever there is no transport, which is physically consistent. Typical values for gravel beds: $\mu = 0.3$–$1.5$; **smaller $\mu$ means stronger smoothing**.

Enable it at instantiation:

```python
rbd = RiverBedDynamics(
    rmg, ...
    use_bed_diffusion=True,
    bed_diffusion_mu=0.5,       # default — adjust to calibrate
    # bed_diffusion_mode='nonlinear',  # default
    # bed_diffusion_mode='constant',   # alternative: fixed D everywhere
    # bed_diffusion_coeff=0.005,       # D value for constant mode
)
```

The cell below re-runs the same one-day simulation with diffusion enabled and compares the longitudinal profiles.

In [ ]:
# Re-initialise grid and components for a clean diffusion run
t_d, t0_d = 0, 0

with open(zDEM) as f:
    rmg_d = load_esri_ascii(f, name="topographic__elevation")
z_d = rmg_d.at_node["topographic__elevation"]
rmg_d.add_zeros("surface_water__depth", at="node")
rmg_d["node"]["topographic__elevation_original"] = copy.deepcopy(z_d)

of_d = OverlandFlow(rmg_d, h_init=0.001, mannings_n=n, rainfall_intensity=0.0)
of_d._rainfall_intensity = np.zeros_like(z_d, dtype=float)
of_d._rainfall_intensity[in_n] = 0.02

rmg_d.add_zeros("surface_water__velocity", at="node")
rmg_d.add_zeros("surface_water__velocity", at="link")
rmg_d["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
    rmg_d, "surface_water__depth"
)
rmg_d.set_watershed_boundary_condition_outlet_id([1, 2], z_d, 45.0)

fixed_d = np.zeros_like(z_d)
fixed_d[fixed_nodes_id] = 1
qb_d = np.full(rmg_d.number_of_links, 0.0)
qb_d[in_l] = in_qb

calc_node_id_d = np.reshape(
    np.arange(128, 136),
    (int(np.arange(128, 136).shape[0] / rmg_d.number_of_node_columns),
     rmg_d.number_of_node_columns)
)

rbd_d = RiverBedDynamics(
    rmg_d,
    gsd=gsd,
    outlet_boundary_condition="fixedValue",
    bed_surf__elev_fix_node=fixed_d,
    sed_transp__bedload_rate_fix_link=qb_d,
    check_advective_cfl=False,
    use_bed_diffusion=True,    # ← NEW: enable gravitational diffusion
    bed_diffusion_mu=0.5,      # ← Talmon parameter: smaller = more smoothing
)

print("Diffusion run instantiated.")
print(f"  bed_diffusion_mode : {rbd_d._bed_diffusion_mode}")
print(f"  bed_diffusion_mu   : {rbd_d._bed_diffusion_mu}")

In [ ]:
# Spin-up
while t0_d < sim_max_t_0:
    of_d.overland_flow(dt=max_dt)
    rmg_d["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg_d, "surface_water__depth"
    )
    t0_d += max_dt

rmg_d["link"]["surface_water__velocity"] = (
    rmg_d["link"]["surface_water__discharge"] / rmg_d["link"]["surface_water__depth"]
)

# Main loop
save_dti_d = copy.deepcopy(save_data_time_interval)
save_dti_d_orig = save_dti_d
progress0 = 0
while t_d < sim_max_t:
    rbd_d._surface_water__velocity_prev_time_link = (
        rmg_d["link"]["surface_water__discharge"] / rmg_d["link"]["surface_water__depth"]
    )
    of_d.overland_flow(dt=max_dt)
    rmg_d["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg_d, "surface_water__depth"
    )
    rmg_d["link"]["surface_water__velocity"] = (
        rmg_d["link"]["surface_water__discharge"] / rmg_d["link"]["surface_water__depth"]
    )
    rbd_d._grid._dt = max_dt
    rbd_d.run_one_step()
    for i in calc_node_id_d:
        rmg_d["node"]["topographic__elevation"][i] = (
            rmg_d["node"]["topographic__elevation"][i - rmg_d.number_of_node_columns]
        )
    t_d += max_dt
    progress = int((t_d / sim_max_t) * 100)
    if progress > progress0 + 1:
        print(f"\rDiffusion run: [{progress}%]", end="")
        progress0 = progress

print("\nDone.")

In [ ]:
# Compare baseline vs diffusion profile
n_col_d = rmg_d.number_of_node_columns
n_row_d = rmg_d.number_of_node_rows

z_base_profile = np.reshape(
    rmg["node"]["topographic__elevation"], (n_row, n_col)
)[1:-2, 1]
z_diff_profile = np.reshape(
    rmg_d["node"]["topographic__elevation"], (n_row_d, n_col_d)
)[1:-2, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(x, z0, "k-", lw=1.5, label="Initial")
ax.plot(x, z_base_profile, "b-",  lw=2, label="Baseline (no diffusion)")
ax.plot(x, z_diff_profile, "g--", lw=2, label="With diffusion (μ=0.5)")
ax.plot(x, zAnalytic,      "r:",  lw=1.5, label="Analytical")
ax.set_xlim(0, 1500)
ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("Bed elevation [m]")
ax.set_title("Longitudinal profile: baseline vs diffusion")
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(x, z_diff_profile - z_base_profile, "g-", lw=2)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("Elevation difference [m]")
ax.set_title("Diffusion effect (diffusion − baseline)")
plt.tight_layout()
plt.show()

print("Diffusion CFL limit at end of run:")
print(f"  {rbd_d.calc_max_stable_dt_diffusive(safety=0.5):.2f} s  "
      f"(dt used = {max_dt} s)")

---
## 7. Higher-Accuracy Time Integration with RK2

The default `time_stepping="euler"` is first-order accurate in time. The new `time_stepping="rk2"` option uses Heun's method (predictor-corrector), which is **second-order accurate**: halving dt reduces the temporal truncation error by a factor of four instead of two.

**Cost:** RK2 requires two bedload evaluations per step instead of one. However, for the same accuracy budget you can use ~2× the stable dt with RK2, so the net computational cost is comparable to Euler at half the dt.

**When to prefer RK2:**
- Long multi-day simulations where temporal accuracy matters.
- When comparing against analytical solutions or published data.
- When you want to use a larger dt without sacrificing accuracy.

The hydraulics (shear stress, water depth) are held fixed during the two predictor-corrector stages; only the bed elevation changes between them.

In [ ]:
# Re-initialise for a clean RK2 run
t_r, t0_r = 0, 0

with open(zDEM) as f:
    rmg_r = load_esri_ascii(f, name="topographic__elevation")
z_r = rmg_r.at_node["topographic__elevation"]
rmg_r.add_zeros("surface_water__depth", at="node")
rmg_r["node"]["topographic__elevation_original"] = copy.deepcopy(z_r)

of_r = OverlandFlow(rmg_r, h_init=0.001, mannings_n=n, rainfall_intensity=0.0)
of_r._rainfall_intensity = np.zeros_like(z_r, dtype=float)
of_r._rainfall_intensity[in_n] = 0.02

rmg_r.add_zeros("surface_water__velocity", at="node")
rmg_r.add_zeros("surface_water__velocity", at="link")
rmg_r["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
    rmg_r, "surface_water__depth"
)
rmg_r.set_watershed_boundary_condition_outlet_id([1, 2], z_r, 45.0)

fixed_r = np.zeros_like(z_r)
fixed_r[fixed_nodes_id] = 1
qb_r = np.full(rmg_r.number_of_links, 0.0)
qb_r[in_l] = in_qb

calc_node_id_r = np.reshape(
    np.arange(128, 136),
    (int(np.arange(128, 136).shape[0] / rmg_r.number_of_node_columns),
     rmg_r.number_of_node_columns)
)

rbd_r = RiverBedDynamics(
    rmg_r,
    gsd=gsd,
    outlet_boundary_condition="fixedValue",
    bed_surf__elev_fix_node=fixed_r,
    sed_transp__bedload_rate_fix_link=qb_r,
    check_advective_cfl=False,
    time_stepping="rk2",       # ← NEW: Heun's method (2nd-order accurate)
)

print(f"Time stepping scheme: {rbd_r._time_stepping}")

In [ ]:
# Spin-up
while t0_r < sim_max_t_0:
    of_r.overland_flow(dt=max_dt)
    rmg_r["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg_r, "surface_water__depth"
    )
    t0_r += max_dt

rmg_r["link"]["surface_water__velocity"] = (
    rmg_r["link"]["surface_water__discharge"] / rmg_r["link"]["surface_water__depth"]
)

# Main loop
progress0 = 0
while t_r < sim_max_t:
    rbd_r._surface_water__velocity_prev_time_link = (
        rmg_r["link"]["surface_water__discharge"] / rmg_r["link"]["surface_water__depth"]
    )
    of_r.overland_flow(dt=max_dt)
    rmg_r["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg_r, "surface_water__depth"
    )
    rmg_r["link"]["surface_water__velocity"] = (
        rmg_r["link"]["surface_water__discharge"] / rmg_r["link"]["surface_water__depth"]
    )
    rbd_r._grid._dt = max_dt
    rbd_r.run_one_step()
    for i in calc_node_id_r:
        rmg_r["node"]["topographic__elevation"][i] = (
            rmg_r["node"]["topographic__elevation"][i - rmg_r.number_of_node_columns]
        )
    t_r += max_dt
    progress = int((t_r / sim_max_t) * 100)
    if progress > progress0 + 1:
        print(f"\rRK2 run: [{progress}%]", end="")
        progress0 = progress

print("\nDone.")

In [ ]:
# Compare Euler vs RK2 profiles
n_col_r = rmg_r.number_of_node_columns
n_row_r = rmg_r.number_of_node_rows
z_rk2_profile = np.reshape(
    rmg_r["node"]["topographic__elevation"], (n_row_r, n_col_r)
)[1:-2, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(x, z0, "k-", lw=1.5, label="Initial")
ax.plot(x, z_base_profile, "b-",  lw=2, label="Euler (default)")
ax.plot(x, z_rk2_profile,  "m--", lw=2, label="RK2")
ax.plot(x, zAnalytic,      "r:",  lw=1.5, label="Analytical")
ax.set_xlim(0, 1500)
ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("Bed elevation [m]")
ax.set_title("Longitudinal profile: Euler vs RK2")
ax.legend(fontsize=9)

ax = axes[1]
dz_rk2 = z_rk2_profile - z_base_profile
ax.bar(x, dz_rk2, width=x[1] - x[0], color="mediumpurple", alpha=0.7)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("RK2 − Euler elevation [m]")
ax.set_title(f"Scheme difference (max = {np.abs(dz_rk2).max():.4f} m)")
plt.tight_layout()
plt.show()

---
## 8. Post-Run Diagnostics

After any run you can inspect diagnostic attributes to assess numerical health:

| Attribute | Description |
|---|---|
| `rbd._bed_surf__gsd_residual_max` | Max $\|\sum f_i - 1\|$ before last GSD renormalisation |
| `rbd._bed_surf__gsd_residual_mean` | Mean of the same residual |
| `rbd.calc_max_stable_dt_advective()` | Advective CFL limit [s] |
| `rbd.calc_max_stable_dt_diffusive()` | Diffusive CFL limit [s] |
| `rbd.calc_max_stable_dt()` | Combined limit — use this to choose dt |

The GSD residual measures how far the grain-size fractions drifted from summing to 1.0 before renormalisation. Values close to zero are ideal; values above `1e-3` (the default warning threshold) indicate numerical drift.

In [ ]:
for label, r in [("Baseline (Euler)", rbd),
                  ("Diffusion (Euler)", rbd_d),
                  ("RK2",               rbd_r)]:
    print(f"=== {label} ===")
    print(f"  GSD residual max  : {r._bed_surf__gsd_residual_max:.2e}")
    print(f"  GSD residual mean : {r._bed_surf__gsd_residual_mean:.2e}")
    dt_adv  = r.calc_max_stable_dt_advective(safety=0.5)
    dt_diff = r.calc_max_stable_dt_diffusive(safety=0.5)
    dt_safe = r.calc_max_stable_dt(safety=0.5)
    print(f"  CFL advective (safety=0.5) : {dt_adv:.2f} s")
    print(f"  CFL diffusive (safety=0.5) : {dt_diff:.2f} s")
    print(f"  Recommended dt (safety=0.5): {dt_safe:.2f} s  "
          f"[dt used = {max_dt} s]")
    status = "✅" if max_dt <= dt_safe else "⚠️ "
    print(f"  {status}  dt is {'within' if max_dt <= dt_safe else 'ABOVE'} the CFL limit")
    print()

---
## Summary

| Feature | How to enable | When to use |
|---|---|---|
| Link depth synchronisation | `map_mean_of_link_nodes_to_link` after every `of.overland_flow()` | **Always required** in coupled runs |
| CFL analysis | `rbd.calc_max_stable_dt(safety=0.5)` | To choose a safe dt before a long run |
| Gravitational diffusion | `use_bed_diffusion=True`, `bed_diffusion_mu=0.5` | Smooth staircase artefacts on square grids |
| RK2 integrator | `time_stepping="rk2"` | Higher accuracy at same dt; good for long runs |
| Post-run health check | `rbd._bed_surf__gsd_residual_max` | Verify numerical stability after any run |

### Click here for more [Landlab tutorials](https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html)